# AI6130 Assignment 2
Upload the provided workflow archive. No paid API or API key is needed. Run the smoke tests before starting formal experiments. Results are written under GPU/runs; download them before the Colab runtime expires.

In [ ]:
from google.colab import files
from pathlib import Path
import tarfile, os
uploaded=files.upload()
archive=next(name for name in uploaded if name.endswith('.tar.gz'))
root=Path('/content/AI6130-Ass2'); root.mkdir(exist_ok=True)
with tarfile.open(archive) as source:
    source.extractall(root,filter='data')
os.chdir(root/'GPU')
print(Path.cwd())


In [ ]:
import sys, subprocess
assert (3,10) <= sys.version_info[:2] <= (3,12), 'Use a Python 3.10-3.12 runtime for these pins'
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-compatible.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-deps','../AI6130_Assignment2/peft'],check=True)


On Ampere or newer GPUs, use the BF16 profile. T4 selects a full-FP32 fallback that needs its own successful smoke test. If a smoke test runs out of memory, stop and use the NSCC workflow rather than reporting partial experiments. Keep the same selected profile for all four cells.

In [ ]:
import torch,json,os
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
if not torch.cuda.is_bf16_supported():
    assert not any(Path('runs').glob('main-*')) if Path('runs').exists() else True
    Path('configs/main.json').write_text(Path('configs/t4-fp32.json').read_text())
os.environ['HF_HOME']=str(Path('.cache/huggingface').resolve())
os.environ['TOKENIZERS_PARALLELISM']='false'
def run(*args):
    subprocess.run([sys.executable,*args],check=True)
run('tools/audit.py')
run('tools/test_protocol.py')
run('tools/freeze_models.py')
run('tools/download_models.py')
run('tools/smoke.py')


In [ ]:
for model in ['tinyllama','openllama']:
    for method in ['lora','adapterp']:
        run('tools/train.py','--model',model,'--method',method,'--smoke','--resume')
        for task in ['AQuA','AddSub']:
            run('tools/evaluate.py','--run',f'runs/smoke-{model}-{method}-2ep','--task',task,'--limit','2')


Formal experiments: these may take longer than one free Colab allocation. Preserve complete run directories and reload them before rerunning with --resume. A saved adapter alone is not an optimizer checkpoint. Keep active files on local disk and copy checkpoints to Drive periodically, not after every prediction.

In [ ]:
for model in ['tinyllama','openllama']:
    for method in ['lora','adapterp']:
        run('tools/train.py','--model',model,'--method',method,'--resume')
        for task in ['AQuA','AddSub']:
            run('tools/evaluate.py','--run',f'runs/main-{model}-{method}-2ep','--task',task)
run('tools/summarize.py')
print(Path('results/summary.md').read_text())


In [ ]:
import shutil
shutil.make_archive('/content/AI6130-Ass2-G2605698H-results','zip','.', 'runs')
files.download('/content/AI6130-Ass2-G2605698H-results.zip')
